# Hybrid Forecast Model Analysis: Prophet + LSTM

This notebook analyzes the results of our hybrid forecasting model that combines:
1. Facebook Prophet for trend/seasonality
2. LSTM on residuals with GARCH volatility features
3. Combined predictions for final forecasts

We'll load the saved models and analyze:
- Prophet components (trend, seasonality)
- LSTM residual patterns
- Feature importance
- Interactive forecast visualization

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from prophet import Prophet
import tensorflow as tf
from tensorflow import keras
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime, timedelta

# Local imports
from data_loader import download_data, prepare_prophet_df
from garch import fit_garch11, forecast_volatility

# Set style
plt.style.use('seaborn')
sns.set_palette("husl")

## 1. Load Models and Data

Let's load our saved models and the original training data to analyze components and performance.

In [ ]:
# Load saved models
with open('models/prophet_model.pkl', 'rb') as f:
    prophet_model = pickle.load(f)

# Load LSTM and scaler if they exist
try:
    lstm_model = keras.models.load_model('models/lstm_model.keras')
    with open('models/scaler.pkl', 'rb') as f:
        scaler = pickle.load(f)
    print("Loaded both Prophet and LSTM models")
except:
    print("Only Prophet model found")
    lstm_model = None
    scaler = None

# Download historical data (same range as training)
ticker = "BTC-USD"
start_date = "2019-01-01"
end_date = "2023-12-31"
horizon = 30

df = download_data(ticker, start_date, end_date)
df = df[["Close"]].dropna()

# Split into train/test as in training
train = df.iloc[:-horizon].copy()
test = df.iloc[-horizon:].copy()

print(f"Data shape: {df.shape}")
print(f"Train period: {train.index[0]} to {train.index[-1]}")
print(f"Test period: {test.index[0]} to {test.index[-1]}")

## 2. Prophet Components Analysis

Let's analyze Prophet's decomposition of the time series into trend, seasonality, and holidays components.

In [ ]:
# Get Prophet components
prophet_df = prepare_prophet_df(train)
forecast = prophet_model.predict(prophet_df)

# Plot components
prophet_model.plot_components(forecast)
plt.tight_layout()
plt.show()

# Create interactive component plots
components = ['trend', 'weekly', 'yearly']
fig = go.Figure()

for component in components:
    if component in forecast.columns:
        fig.add_trace(go.Scatter(
            x=forecast['ds'],
            y=forecast[component],
            name=component.capitalize(),
            mode='lines'
        ))

fig.update_layout(
    title=f"{ticker} - Prophet Components",
    xaxis_title="Date",
    yaxis_title="Component Value",
    hovermode='x unified'
)
fig.show()

## 3. LSTM Analysis

Let's analyze the LSTM's contribution by looking at residual patterns and feature importance.

In [ ]:
# Compute in-sample residuals
train_prophet_preds = prophet_model.predict(prophet_df)["yhat"].values
residuals = train['Close'].values.flatten() - train_prophet_preds

# Compute volatility feature
train_returns = train['Close'].pct_change()
vol = train_returns.rolling(window=5).std().fillna(method='bfill').values

# Plot residuals vs volatility
plt.figure(figsize=(12, 6))
plt.subplot(2,1,1)
plt.plot(train.index, residuals, label='Prophet Residuals')
plt.title('Prophet Residuals Over Time')
plt.legend()

plt.subplot(2,1,2)
plt.plot(train.index, vol, label='Rolling Volatility', color='orange')
plt.title('5-day Rolling Volatility')
plt.legend()
plt.tight_layout()
plt.show()

# Interactive residuals analysis
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=train.index,
    y=residuals,
    name='Residuals',
    mode='lines',
    line=dict(color='blue')
))
fig.add_trace(go.Scatter(
    x=train.index,
    y=vol,
    name='Volatility',
    mode='lines',
    line=dict(color='red'),
    yaxis='y2'
))

fig.update_layout(
    title='Prophet Residuals vs Volatility',
    xaxis_title='Date',
    yaxis_title='Residual Value',
    yaxis2=dict(
        title='Volatility',
        overlaying='y',
        side='right'
    ),
    hovermode='x unified'
)
fig.show()

## 4. Combined Forecast Analysis

Now let's analyze the final hybrid predictions and compare with individual components.

In [ ]:
# Generate forecasts
future = prophet_model.make_future_dataframe(periods=horizon, freq='D')
prophet_forecast = prophet_model.predict(future)
prophet_pred = prophet_forecast['yhat'].values[-horizon:]

# If we have LSTM, generate residual predictions
if lstm_model is not None:
    # Prepare sequences for LSTM (last seq_len values)
    seq_len = 30  # from training
    last_res = residuals[-seq_len:]
    last_vol = vol[-seq_len:]
    
    # Generate residual predictions
    preds_res = []
    cur_res_seq = last_res.copy()
    cur_vol_seq = last_vol.copy()
    
    for h in range(horizon):
        feat = np.vstack([cur_res_seq, cur_vol_seq]).T.reshape(1, seq_len, 2)
        feat_scaled = scaler.transform(feat.reshape(-1, 2)).reshape(1, seq_len, 2)
        pred_res = lstm_model.predict(feat_scaled, verbose=0)[0, 0]
        preds_res.append(pred_res)
        
        # Roll sequences
        cur_res_seq = np.roll(cur_res_seq, -1)
        cur_res_seq[-1] = pred_res
        cur_vol_seq = np.roll(cur_vol_seq, -1)
        cur_vol_seq[-1] = cur_vol_seq[-2]  # simple carry forward
    
    hybrid_pred = prophet_pred + np.array(preds_res)
else:
    hybrid_pred = prophet_pred

# Plot results
plt.figure(figsize=(12, 6))
plt.plot(test.index, test['Close'], label='Actual', color='black')
plt.plot(test.index, prophet_pred, label='Prophet', color='blue', linestyle='--')
if lstm_model is not None:
    plt.plot(test.index, hybrid_pred, label='Hybrid', color='red')
plt.title(f'{ticker} - Forecast Comparison')
plt.legend()
plt.tight_layout()
plt.show()

# Interactive forecast plot
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=test.index,
    y=test['Close'],
    name='Actual',
    mode='lines',
    line=dict(color='black')
))

fig.add_trace(go.Scatter(
    x=test.index,
    y=prophet_pred,
    name='Prophet',
    mode='lines',
    line=dict(color='blue', dash='dash')
))

if lstm_model is not None:
    fig.add_trace(go.Scatter(
        x=test.index,
        y=hybrid_pred,
        name='Hybrid',
        mode='lines',
        line=dict(color='red')
    ))

fig.update_layout(
    title=f'{ticker} - Forecast Comparison',
    xaxis_title='Date',
    yaxis_title='Price',
    hovermode='x unified'
)
fig.show()

# Print metrics
from utils import rmse, mape, directional_accuracy

print("\nMetrics:")
print(f"Prophet RMSE: {rmse(test['Close'].values, prophet_pred):.2f}")
print(f"Prophet MAPE: {mape(test['Close'].values, prophet_pred):.2f}%")
print(f"Prophet Dir. Accuracy: {directional_accuracy(test['Close'].values, prophet_pred):.2f}%")

if lstm_model is not None:
    print(f"\nHybrid RMSE: {rmse(test['Close'].values, hybrid_pred):.2f}")
    print(f"Hybrid MAPE: {mape(test['Close'].values, hybrid_pred):.2f}%")
    print(f"Hybrid Dir. Accuracy: {directional_accuracy(test['Close'].values, hybrid_pred):.2f}%")

## 5. Conclusion and Recommendations

Based on the analysis above, we can observe:
1. Prophet captures overall trend and seasonality well
2. LSTM helps capture residual patterns when volatility is high
3. Directional accuracy needs improvement - consider:
   - Adding more features (technical indicators)
   - Longer training epochs
   - Tuning LSTM architecture
   - Ensemble with other models